In [1]:
import re
import sys
import os
import cv2
import numpy as np

from marsneuralzoo.models.e2emvm import E2emvm
from marsneuralzoo.models.yolo_seg import SegmentationYolo

e2emvm = E2emvm(multiview=False)
seg_yo = SegmentationYolo()

Loaded SuperPoint model


In [2]:
import os
import sys
import cv2
import itertools

# from dataengine.generator.obstacle.mask import (
#     remove_overlapped_area,
# )

debug_image_dir = './cache/debug/5'

image_name_list = os.listdir(debug_image_dir)
image_name_list.sort(key=lambda x: int(os.path.splitext(x)[0]))

image_list = []
for image_name in image_name_list:
    image_path = os.path.join(debug_image_dir, image_name)
    image = cv2.imread(image_path)
    image_list.append(image)


def split_list(input_list, chunk_size):
    """input_list를 chunk_size 크기만큼 나눠 분할된 리스트들의 리스트로 반환"""
    return [
        input_list[i : i + chunk_size]
        for i in range(0, len(input_list), chunk_size)
    ]


chunk_size = 10
split_image_lists = split_list(image_list, chunk_size)

seg_results_list = []

for split_image_list in split_image_lists:
    results = seg_yo.segment_batch(split_image_list)
    seg_results_list.extend(results)

# seg_results_list = seg_yo.segment_ba                tch(image_list)


def remove_overlapped_area(segs, confs):
    """
    Remove intersection area based on confidences
    """
    indices = range(len(segs))
    combinations = list(itertools.combinations(indices, 2))
    for i, j in combinations:
        intersection = segs[i] & segs[j] > 0
        if np.sum(intersection) > 0:
            if confs[i] > confs[j]:
                segs[j][intersection] = 0
            else:
                segs[i][intersection] = 0

    mask = np.sum(segs, axis=(1, 2)) >= 100
    filtered_segs = segs[mask]
    return filtered_segs


raw_segs_list = []
segs_list = []
for seg_results in seg_results_list:
    masks = seg_results['masks']
    confs = seg_results['scores']
    raw_segs_list.append(masks)
    masks = remove_overlapped_area(masks, confs)
    segs_list.append(masks)

Loading /media/vol/shared/obstacle/models/yolosegment/best.torchscript for TorchScript inference...


In [3]:
from dataengine.generator.obstacle.debug_utils import draw_mask

In [4]:
# idx = 76

# for segs in raw_segs_list[idx]:

#     debug = image_list[idx].copy()
#     draw_mask(debug, segs)
#     cv2.imshow("", debug)
#     cv2.waitKey(0)

# cv2.destroyAllWindows()

In [5]:
out_dir = './cache/debug'
IMG_H, IMG_W = image_list[0].shape[:2]

video_writer = cv2.VideoWriter(
    f"{out_dir}/input.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (IMG_W, IMG_H),
)

frame_count = 0
for image, segs in zip(image_list, raw_segs_list):
    idx = 0
    frame = image.copy()
    for seg in segs:
        draw_mask(frame, seg, idx, 0.5)
        idx += 1

    cv2.putText(
        frame,
        f"{frame_count:03d}",
        (40, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        1,
        cv2.LINE_AA,
    )
    video_writer.write(frame)
    frame_count += 1

video_writer.release()

In [6]:
from dataengine.generator.obstacle.debug_utils import (
    draw_mask,
    visualize_optical_flow,
    Statistics,
)

from dataengine.generator.obstacle.segment_tracker import (
    SegmentTracklet,
)
from itertools import count

from typing import Dict, List, Optional

# from dataengine.generator.obstacle.assignment import assign_segments
from dataengine.generator.obstacle.assignment import calc_segment_iou


def id_image_to_debug_image(image, id_image):
    ids = np.unique(id_image.flatten())

    for id in ids:
        if id == -1:
            continue
        mask = (id_image == id).astype(np.uint8)
        draw_mask(image, mask, id, 0.8)


id_video_writer = cv2.VideoWriter(
    f"{out_dir}/id_debug.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (IMG_W, IMG_H),
)

missed_id_video_writer = cv2.VideoWriter(
    f"{out_dir}/missed_id_debug.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (IMG_W, IMG_H),
)


flow_video_writer = cv2.VideoWriter(
    f"{out_dir}/flow.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (IMG_W, IMG_H),
)

from lapsolver import solve_dense

np.set_printoptions(precision=3, suppress=True)


def assign_segments(segs_a, segs_b, min_iou_thresh):
    """
    Find an optimal assignment between two sets of segment masks.

    Given two arrays of segment masks, use linear assignment to find an optimal
    assignment between them, such that sum(IOU) of all assignments is maximized.
    Additionally, a minimum IOU threshold is applied to all valid assignments.
    An assignment that does not pass this threshold is considered invalid,
    and elements that are not part of any valid assignment are considered "unmatched".

    :param segs_a: shape == (n, h, w)
    :param segs_b: shape == (m, h, w)
    :param min_iou_thresh: scalar, min iou threshold for assignment
    :return:
    """
    rows = segs_a.shape[0]
    cols = segs_b.shape[0]

    iou_mat = np.zeros((rows, cols))
    for r in range(rows):
        for c in range(cols):
            iou_mat[r, c] = calc_segment_iou(segs_a[r], segs_b[c])

    c_overlap_indice_set = set()
    for r in range(iou_mat.shape[0]):
        if np.sum(iou_mat[r] > min_iou_thresh) < 2:
            continue

        overlap_indices = np.where(iou_mat[r] > min_iou_thresh)
        for c in overlap_indices[0]:
            c_overlap_indice_set.add((r, c))

    r_overlap_indice_set = set()
    for c in range(iou_mat.shape[1]):
        if np.sum(iou_mat[:, c] > min_iou_thresh) < 2:
            continue
        overlap_indices = np.where(iou_mat[:, c] > min_iou_thresh)
        for r in overlap_indices[0]:
            r_overlap_indice_set.add((r, c))
    matched_indices = np.array(solve_dense(-iou_mat)).T

    unmatched_segs_a = []
    for idx in range(rows):
        if idx not in matched_indices[:, 0]:
            unmatched_segs_a.append(idx)

    unmatched_segs_b = []
    for idx in range(cols):
        if idx not in matched_indices[:, 1]:
            unmatched_segs_b.append(idx)

    matches = []
    for m in matched_indices:
        if iou_mat[m[0], m[1]] < min_iou_thresh:
            unmatched_segs_a.append(m[0])
            unmatched_segs_b.append(m[1])
        else:
            matches.append(m)

    matches = np.array(matches)

    hit_mat = (iou_mat > min_iou_thresh).astype(np.uint8)

    matched_mat = np.zeros((rows, cols), dtype=np.uint8)

    for r, c in matches:
        matched_mat[r, c] = 1

    overlap_indices = np.where(hit_mat - matched_mat)
    overlap_indices = list(zip(overlap_indices[0], overlap_indices[1]))

    out_c_overlap_indice_set = []
    out_r_overlap_indice_set = []
    for overlap_indice in overlap_indices:

        if overlap_indice in c_overlap_indice_set:
            out_c_overlap_indice_set.append(overlap_indice)
        if overlap_indice in r_overlap_indice_set:
            out_r_overlap_indice_set.append(overlap_indice)

    return (
        matches,
        unmatched_segs_a,
        unmatched_segs_b,
        out_c_overlap_indice_set,
        out_r_overlap_indice_set,
    )


class SegmentTracker:
    """
    This class tracks the segments detected by segment model using optical flow.
    """

    def __init__(
        self,
        cam_index: int,
        id_tracklet_dict: Dict[int, SegmentTracklet],
        max_missed_track=12,
        min_iou_thresh=0.3,
    ):
        """Initialises states with tracklet_database."""
        self.cam_index = cam_index
        self.prev_timestamp = -1
        self.prev_gray: Optional[np.ndarray] = None  # H x W

        # id_tracklet_dict[id] -> tracklet
        self.id_tracklet_dict = id_tracklet_dict

        # missed_tracklets[id] -> tracklet
        self.missed_tracklets: Dict[int, SegmentTracklet] = {}

        # tracked_tracklets[id] -> tracklet
        self.tracked_tracklets: Dict[int, SegmentTracklet] = {}
        self.grid = None

        self.max_missed_track = max_missed_track
        self.min_iou_thresh = min_iou_thresh

    def track(self, timestamp, image, segs):
        """Takes a new camera measurements and tracks currently activated
        segment tracklets.
        :param timestamp: timestamp
        :param image: image
        :param segs: segment masks from deep model
        :return: recent tracklet ids.
        """

        h, w = image.shape[:2]
        shape = image.shape[:2]

        Statistics.startTimer('SegmentTracker_flow')
        curr_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        curr_gray = cv2.resize(curr_gray, (w // 2, h // 2))

        if self.prev_gray is None:
            self.prev_timestamp = timestamp
            self.prev_gray = curr_gray

            h, w = image.shape[:2]
            remap = np.meshgrid(
                np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32)
            )
            self.grid = np.stack(remap, axis=2)

        flow = self._calculate_flow(self.prev_gray, curr_gray)

        flow = cv2.resize(flow, (w, h)) * 2

        Statistics.stopTimer('SegmentTracker_flow')
        debug_img = image.copy()
        visualize_optical_flow(debug_img, flow)

        prev_color = self.prev_gray.copy()
        prev_color = cv2.resize(prev_color, (w, h))
        prev_color = cv2.cvtColor(prev_color, cv2.COLOR_GRAY2RGB)
        visualize_optical_flow(prev_color, flow)
        cv2.putText(
            prev_color,
            f"{timestamp:03d}",
            (40, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            1,
            cv2.LINE_AA,
        )
        cv2.putText(
            debug_img,
            f"{timestamp:03d}",
            (40, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            1,
            cv2.LINE_AA,
        )

        flow_video_writer.write(prev_color)
        flow_video_writer.write(debug_img)

        warping_matrix = self.grid - flow

        Statistics.startTimer('SegmentTracker_remap')

        # project active tracklets' ids into image
        tracked_id_image = np.full(shape, -1.0, dtype=np.float32)
        missed_id_image = np.full(shape, -1.0, dtype=np.float32)

        for id, seg_trl in self.tracked_tracklets.items():
            seg = seg_trl.query_segment_mask(
                self.prev_timestamp, self.cam_index
            )
            tracked_id_image[seg > 0] = id

        for id, seg_trl in self.missed_tracklets.items():
            seg = seg_trl.predicted_mask
            missed_id_image[seg > 0] = id

        id_debug = image.copy()
        tracked_id_image_i = tracked_id_image.astype(int)

        prev_color = self.prev_gray.copy()
        prev_color = cv2.resize(prev_color, (w, h))
        prev_color = cv2.cvtColor(prev_color, cv2.COLOR_GRAY2RGB)
        # id_image_to_debug_image(prev_color, tracked_id_image_i)
        id_image_to_debug_image(id_debug, tracked_id_image_i)
        cv2.putText(
            id_debug,
            f"{timestamp:03d}",
            (40, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            2,
            cv2.LINE_AA,
        )
        # id_video_writer.write(prev_color)
        # id_video_writer.write(id_debug)

        tracked_id_image = cv2.remap(
            tracked_id_image,
            warping_matrix,
            None,
            interpolation=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_REFLECT,
        )

        missed_id_image = cv2.remap(
            missed_id_image,
            warping_matrix,
            None,
            interpolation=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_REFLECT,
        )

        Statistics.stopTimer('SegmentTracker_remap')

        tracked_id_image = tracked_id_image.astype(int)
        id_debug = image.copy()

        # prev_color = self.prev_gray.copy()
        # prev_color = cv2.resize(prev_color, (w, h))
        # prev_color = cv2.cvtColor(prev_color, cv2.COLOR_GRAY2RGB)
        # id_image_to_debug_image(prev_color, tracked_id_image)
        id_image_to_debug_image(id_debug, tracked_id_image)
        cv2.putText(
            id_debug,
            f"{timestamp:03d}",
            (40, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            2,
            cv2.LINE_AA,
        )
        # id_video_writer.write(prev_color)
        id_video_writer.write(id_debug)

        tracked_ids = np.unique(tracked_id_image.flatten())
        tracked_ids = tracked_ids[tracked_ids > -1]
        tracked_ids = list(tracked_ids)

        missed_id_image = missed_id_image.astype(int)

        id_debug = image.copy()
        id_image_to_debug_image(id_debug, missed_id_image)
        cv2.putText(
            id_debug,
            f"{timestamp:03d}",
            (40, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            1,
            cv2.LINE_AA,
        )
        missed_id_video_writer.write(id_debug)

        missed_ids = np.unique(missed_id_image.flatten())
        missed_ids = missed_ids[missed_ids > -1]
        missed_ids = list(missed_ids)

        prev_ids = tracked_ids + missed_ids

        Statistics.startTimer('SegmentTracker_assign')
        Statistics.startTimer('SegmentTracker_assign_0')

        segs_len = len(prev_ids)
        expected_segs = np.zeros((segs_len, h, w), dtype=np.uint8)
        missed_offset = 0
        for b, id in enumerate(tracked_ids):
            id_seg = tracked_id_image == id
            expected_segs[b] = id_seg
            missed_offset += 1

        for b, id in enumerate(missed_ids):
            id_seg = missed_id_image == id
            expected_segs[missed_offset + b] = id_seg
        # matched_idx_pair, missed_idx, unmatched_seg_idx = assign_segments(
        #     expected_segs, segs, self.min_iou_thresh
        # )
        # print(f'time {timestamp} : {tracked_ids}, {missed_ids}')
        Statistics.stopTimer('SegmentTracker_assign_0')
        Statistics.startTimer('SegmentTracker_assign_1')

        half_exp_segs = expected_segs[:, ::2, ::2]
        half_segs = segs[:, ::2, ::2]

        (
            matched_idx_pair,
            missed_idx,
            unmatched_seg_idx,
            c_overlapped_indices,
            r_overlapped_indices,
        ) = assign_segments(half_exp_segs, half_segs, self.min_iou_thresh)

        for r, c in c_overlapped_indices:
            keep = segs[c]
            remove = expected_segs[r]

            inter = (remove & keep).astype(bool)
            segs[c][inter] = 0

        for r, c in r_overlapped_indices:
            keep = expected_segs[r]
            remove = segs[c]

            inter = (remove & keep).astype(bool)
            remove[inter] = 0

        Statistics.stopTimer('SegmentTracker_assign_1')

        Statistics.stopTimer('SegmentTracker_assign')

        Statistics.startTimer('SegmentTracker_results')

        out_tracklet_ids = []
        self.missed_tracklets = {}
        self.tracked_tracklets = {}
        for id_idx, seg_idx in matched_idx_pair:
            id = prev_ids[id_idx]
            seg = segs[seg_idx]
            seg_trl = self.id_tracklet_dict[id]

            seg_trl.add_cam_segment_mask(timestamp, self.cam_index, seg)
            out_tracklet_ids.append(id)
            self.tracked_tracklets[id] = seg_trl

        for id_idx in missed_idx:
            id = prev_ids[id_idx]
            seg_trl = self.id_tracklet_dict[id]
            seg_trl.missed_count += 1

            if seg_trl.missed_count < self.max_missed_track:
                self.missed_tracklets[id] = seg_trl
                seg_trl.predicted_mask = expected_segs[id_idx]

        for seg_idx in unmatched_seg_idx:
            seg = segs[seg_idx]
            seg_trl = SegmentTracklet()
            self.id_tracklet_dict[seg_trl.id] = seg_trl
            seg_trl.add_cam_segment_mask(timestamp, self.cam_index, seg)
            out_tracklet_ids.append(seg_trl.id)

            self.tracked_tracklets[seg_trl.id] = seg_trl

        self.prev_timestamp = timestamp
        self.prev_gray = curr_gray
        Statistics.stopTimer('SegmentTracker_results')

        return out_tracklet_ids

    def _calculate_flow(self, src_gray, dst_gray):

        flow = cv2.calcOpticalFlowFarneback(
            src_gray, dst_gray, None, 0.5, 4, 30, 3, 5, 1.2, 0
        )
        return flow
        padding = 10
        constant_value = 0

        padded_gray1 = cv2.copyMakeBorder(
            src_gray,
            padding,
            padding,
            padding,
            padding,
            cv2.BORDER_CONSTANT,
            value=constant_value,
        )
        padded_gray2 = cv2.copyMakeBorder(
            dst_gray,
            padding,
            padding,
            padding,
            padding,
            cv2.BORDER_CONSTANT,
            value=constant_value,
        )

        flow = cv2.calcOpticalFlowFarneback(
            padded_gray1, padded_gray2, None, 0.5, 5, 50, 3, 4, 1.2, 0
        )
        return flow[padding:-padding, padding:-padding]


_cam_to_trl_ids_dict = {}
_trl_id_to_trl_dict = {}

SegmentTracklet.id_counter = count(0)

idx0 = 0
idx1 = 1

tracker0 = SegmentTracker(idx0, _trl_id_to_trl_dict)
tracker1 = SegmentTracker(idx1, _trl_id_to_trl_dict)

target_from = 0
target_end = 200

if target_end > len(image_list):
    target_end = len(image_list)

out_dir = './cache/debug'
video_writer = cv2.VideoWriter(
    f"{out_dir}/0.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (IMG_W, IMG_H),
)

timestamp = target_from
for i in range(target_from, target_end):
    # if i == 76:
    # continue
    print(f'processing {i}...')
    image = image_list[i]
    segs = segs_list[i]
    curr_tracklet_ids = tracker0.track(timestamp, image, segs)

    frame = image.copy()
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    for id in curr_tracklet_ids:
        mask = _trl_id_to_trl_dict[id].query_segment_mask(timestamp, idx0)
        draw_mask(frame, mask, id, 0.8)
    cv2.putText(
        frame,
        f"{timestamp:03d}",
        (40, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        1,
        cv2.LINE_AA,
    )

    video_writer.write(frame)
    timestamp += 1
video_writer.release()

flow_video_writer.release()
id_video_writer.release()
missed_id_video_writer.release()
# video_writer = cv2.VideoWriter(
#     f"{out_dir}/1.mp4",
#     cv2.VideoWriter_fourcc(*'mp4v'),
#     20,
#     (IMG_W, IMG_H),
# )

# timestamp = target_from
# for i in reversed(range(target_from, target_end)):
#     image = image_list[i]
#     segs = segs_list[i]
#     curr_tracklet_ids = tracker1.track(timestamp, image, segs)

#     frame = image.copy()
#     frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

#     for id in curr_tracklet_ids:
#         mask = _trl_id_to_trl_dict[id].query_segment_mask(timestamp, idx1)
#         draw_mask(frame, mask, id, 0.6)
#     cv2.putText(
#         frame,
#         f"{timestamp:03d}",
#         (40, 80),
#         cv2.FONT_HERSHEY_SIMPLEX,
#         1,
#         (0, 255, 255),
#         1,
#         cv2.LINE_AA,
#     )

#     video_writer.write(frame)
#     timestamp += 1


# video_writer.release()

kj/filesystem-disk-unix.c++:1690: warning: PWD environment variable doesn't match current directory; pwd = /home/mars


processing 0...
processing 1...
processing 2...
processing 3...
processing 4...
processing 5...
processing 6...
processing 7...
processing 8...
processing 9...
processing 10...
processing 11...
processing 12...
processing 13...
processing 14...
processing 15...
processing 16...
processing 17...
processing 18...
processing 19...
processing 20...
processing 21...
processing 22...
processing 23...
processing 24...
processing 25...
processing 26...
processing 27...
processing 28...
processing 29...
processing 30...
processing 31...
processing 32...
processing 33...
processing 34...
processing 35...
processing 36...
processing 37...
processing 38...
processing 39...
processing 40...
processing 41...
processing 42...
processing 43...
processing 44...
processing 45...
processing 46...
processing 47...
processing 48...
processing 49...
processing 50...
processing 51...
processing 52...
processing 53...
processing 54...
processing 55...
processing 56...
processing 57...
processing 58...
process